# GaMMA association of the broadband-station ELEP picks (PyOcto cross-check)

Companion to `redone_broadband_associations_11032026_2-9hz_savethis.ipynb`, which associates the same
picks with PyOcto. This notebook runs GaMMA on them instead, set up the same way as the DAS run in
`gamma_phasenet_das_rainier_events946_10292026_savethis.ipynb`, so the two station catalogues can be
compared associator-to-associator with everything else held fixed:

| item | PyOcto notebook | this notebook |
| --- | --- | --- |
| picks | every `*2023*.csv` in `filepath` (ELEP, 2-9 Hz, BH? + HH? runs) | same files, same merge |
| stations | union of the BH? and HH? `get_stations()` inventories | same inventories, same dedupe on station code |
| box | lon -123 to -120, lat 45 to 48, 0-40 km | same, as `xlim_degree` / `ylim_degree` / `z(km)` |
| velocity model | IASP91 (`data/vmodel/IASP91_1.csv`) via PyOcto 1-D table | same file via GaMMA eikonal, h = 1 km |
| event acceptance, full network | `n_picks=10, n_p_picks=3, n_s_picks=3, n_p_and_s_picks=3` | `min_picks_per_eq=10, min_p_picks_per_eq=3, min_s_picks_per_eq=3` |
| event acceptance, 5 stations | `n_picks=6, n_p_picks=3, n_s_picks=3, n_p_and_s_picks=3` | `min_picks_per_eq=6, min_p_picks_per_eq=3, min_s_picks_per_eq=3` |

GaMMA has no equivalent of PyOcto's `n_p_and_s_picks` (stations with both a P and an S), so that
one condition is not enforced here. Everything GaMMA-specific (BGMM, oversample 5, DBSCAN eps from
`estimate_eps`, sigma limits) is copied from the DAS run. ELEP writes no pick probability, so every
pick gets `prob = 1`.

Run with the `test_environment` kernel (obspy, GaMMA, pyproj).

**Re-running the association cell overwrites `gamma_events.csv` and `gamma_picks.csv` in the results folder.**


In [1]:
import os
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyproj import Proj
from gamma.utils import association, estimate_eps

from obspy import UTCDateTime
from obspy.clients.fdsn.client import Client


## 1. Data

In [2]:
## Paths: where the inputs live and where results and figures go
region = "rainier_seismic_stations"

# Same pick folder as the PyOcto notebook: one NET_STA_YYYYMMDD.csv per station-day from the ELEP runs.
filepath = "/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/seismic-stations_10282026_1/"
pyocto_dir = os.path.join(filepath, "association")          # PyOcto outputs, for the check at the end
data_path = lambda x: os.path.join("data", x)
iasp91_csv = data_path("vmodel/IASP91_1.csv")

result_root = ("/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/"
               f"gamma_association_11022026/{region}")
os.makedirs(result_root, exist_ok=True)
result_path = lambda x: os.path.join(result_root, x)
os.makedirs("figures", exist_ok=True)
figure_dir = lambda x: os.path.join("figures", x)

for name, p in [("picks", filepath), ("pyocto", pyocto_dir), ("iasp91", iasp91_csv), ("results", result_root)]:
    print(f"{'ok' if os.path.exists(p) else 'MISSING':8s}{name:10s}{p}")
print(f"{len(glob.glob(os.path.join(filepath, '*2023*.csv')))} station-day pick files")


ok      picks     /1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/seismic-stations_10282026_1/
ok      pyocto    /1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/seismic-stations_10282026_1/association
ok      iasp91    data/vmodel/IASP91_1.csv
ok      results   /1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/gamma_association_11022026/rainier_seismic_stations
2625 station-day pick files


In [3]:
# Window comes from the DAS campaign index, not a hardcoded pair of dates. These are the real
# first and last files on disk, so the broadband request cannot drift from the DAS coverage it
# will be compared against, and it re-derives itself if the campaign is ever re-indexed.
FILE_INDEX = ('/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/picking-denoising/phasenet-3-month-run_08182026/file_index.csv')
DAS_FILE_SECONDS = 60      # each archive file holds 60 s, so the last file ENDS 60 s after it starts

_idx = pd.read_csv(FILE_INDEX, usecols=['file_time'], parse_dates=['file_time'])

starttime1 = UTCDateTime(_idx.file_time.min().isoformat())
endtime1   = UTCDateTime(_idx.file_time.max().isoformat()) + DAS_FILE_SECONDS

print('DAS index : %d files over %d days' % (len(_idx), _idx.file_time.dt.date.nunique()))
print('starttime1: %s   (first DAS file)' % starttime1)
print('endtime1  : %s   (last DAS file + %d s)' % (endtime1, DAS_FILE_SECONDS))
print('span      : %.2f days' % ((endtime1 - starttime1) / 86400))

# Coverage is not continuous inside that span -- 2023-11-09 has no data at all, and six days
# are partial. Request the whole window, but expect no DAS to associate against on those hours.
_per_day = _idx.file_time.dt.date.value_counts()
print('\npartial days (< 1440 files):')
print(_per_day[_per_day < 1440].sort_index().to_string())

DAS index : 126458 files over 90 days
starttime1: 2023-08-23T19:19:28.000000Z   (first DAS file)
endtime1  : 2023-11-22T00:00:00.000000Z   (last DAS file + 60 s)
span      : 90.19 days

partial days (< 1440 files):
file_time
2023-08-23     281
2023-09-18     471
2023-09-19     664
2023-10-14    1330
2023-11-08    1371
2023-11-10    1380


In [4]:
# Station list for the get_stations() call below, read from the FDSN text file rather than
# retyped into a string -- so adding a station to the file is the only edit needed.
STATION_TXT = 'data/MtRainier_seis.txt'   # '#Network|Station|Lat|Lon|Elev|Sitename|StartTime|EndTime'

# Keep only stations actually operating during the DAS window from the cell above. The file
# carries 1995 NEHRP temporary deployments and stations decommissioned years before this
# campaign (RVC ended 2021-01-10, WPW 2017-04-19); requesting them is silently empty, and they
# would otherwise look like stations that simply produced no picks.
FILTER_TO_DAS_WINDOW = True

_sta = pd.read_csv(STATION_TXT, sep='|')
_sta.columns = [c.lstrip('#').strip() for c in _sta.columns]
_sta['StartTime'] = pd.to_datetime(_sta['StartTime'], errors='coerce')
_sta['EndTime']   = pd.to_datetime(_sta['EndTime'],   errors='coerce')

if FILTER_TO_DAS_WINDOW:
    _t1, _t2 = pd.Timestamp(starttime1.datetime), pd.Timestamp(endtime1.datetime)
    _live = _sta[(_sta['StartTime'] <= _t2) & (_sta['EndTime'] >= _t1)]
else:
    _live = _sta

network = ','.join(sorted(_live['Network'].unique()))
station = ','.join(sorted(_live['Station'].unique()))

print('%s: %d stations, %d live in the DAS window' % (STATION_TXT, len(_sta), len(_live)))
print(_live.groupby('Network')['Station'].count().to_string())
print('\nnetwork = %r' % network)
print('station = %r' % station)

_dropped = _sta[~_sta.index.isin(_live.index)]
if len(_dropped):
    print('\ndropped (not operating in the window):')
    print(_dropped[['Network', 'Station', 'StartTime', 'EndTime']].to_string(index=False))

data/MtRainier_seis.txt: 48 stations, 33 live in the DAS window
Network
CC    25
UW     8

network = 'CC,UW'
station = 'ARAT,CARB,COPP,CRBN,FMW,GNOB,GOBB,GTWY,KAUT,LO2,LON,MILD,OBSR,OPCH,PANH,PARA,PR01,PR02,PR03,PR04,PR05,RCM,RCS,RER,RUSH,SIFT,STAR,STYX,TABR,TAVI,VOIT,WOW,WPEQ'

dropped (not operating in the window):
Network Station  StartTime             EndTime
     CC    SR41 2015-07-09 2018-04-28 00:00:00
     UW     ASF 1995-03-20 1995-06-12 23:59:59
     UW     DLR 1995-04-07 1995-09-19 23:59:59
     UW     FGM 1995-05-20 1995-09-19 23:59:59
     UW     GRL 1995-10-01 1996-07-18 23:59:59
     UW     MHL 1995-08-02 1995-10-31 23:59:59
     UW     RSH 1995-06-13 1995-09-20 23:59:59
     UW     RSU 1999-08-20 2001-09-21 00:00:00
     UW     RVC 1983-01-01 2021-01-10 00:00:00
     UW     RVN 1995-09-26 2002-08-15 00:00:00
     UW     WCR 1995-06-12 1995-09-22 23:59:59
     UW     WI2 1995-09-24 1996-07-17 23:59:59
     UW     WPS 1995-06-29 1995-09-29 23:59:59
     UW     WPW 1980-03

In [5]:
# Station inventories, exactly as in the PyOcto notebook: the picks in filepath were made with
# these same two inventories (BH? and HH?), so the station table built from them matches the
# station codes in the pick files.
client = Client("IRIS")

t1 = starttime1
t2 = endtime1

inventory = client.get_stations(network=network, station = station, channel="BH?",
                                starttime=t1, endtime=t2)

inventory2 = client.get_stations(network=network, station = station, channel="HH?",
                                 starttime=t1, endtime=t2)

In [6]:
## read stations: the UNION of the BH? and HH? inventories, deduped on station code -- the same
## table the PyOcto notebook builds. The picks carry station_code with no network prefix, and no
## code collides across networks, so the bare code is the GaMMA station id.
rows = {}
for chan, inv in (("BH?", inventory), ("HH?", inventory2)):
    n0 = len(rows)
    for net in inv:
        for sta in net:
            rows.setdefault(sta.code, {"id": sta.code,
                                       "longitude": sta.longitude,
                                       "latitude": sta.latitude,
                                       "elevation_m": sta.elevation})
    print(f"{chan}: {len(rows) - n0} stations new to the table")
stations = pd.DataFrame(list(rows.values()), columns=["id", "longitude", "latitude", "elevation_m"])
stations.to_csv(result_path("stations.csv"), index=False)
print(f"{len(stations)} unique stations")
print("Station format:", stations.head(), sep="\n")

## read picks: same merge as the PyOcto notebook (every *2023*.csv in filepath), then one row per
## pick in GaMMA's format: id, timestamp, type, prob. ELEP writes no probability, so prob = 1.
csv_list = sorted(glob.glob(os.path.join(filepath, "*2023*.csv")))
df = pd.concat([pd.read_csv(f, index_col=0) for f in csv_list], axis=0, ignore_index=True)

p = df.loc[df["trace_p_arrival"].notna(), ["station_code", "trace_p_arrival"]].rename(columns={"trace_p_arrival": "timestamp"})
p["type"] = "P"
s = df.loc[df["trace_s_arrival"].notna(), ["station_code", "trace_s_arrival"]].rename(columns={"trace_s_arrival": "timestamp"})
s["type"] = "S"
picks = pd.concat([p, s], ignore_index=True).rename(columns={"station_code": "id"})
picks["timestamp"] = pd.to_datetime(picks["timestamp"], utc=True, format="ISO8601")
picks["prob"] = 1.0
picks = picks.sort_values("timestamp").reset_index(drop=True)

# Picks on stations missing from the table would be dropped silently by GaMMA, as by PyOcto.
missing = sorted(set(picks["id"]) - set(stations["id"]))
print(f"\n{len(picks)} picks from {len(csv_list)} files: " + ", ".join(f"{k} {v}" for k, v in picks["type"].value_counts().items()))
print(f"stations with picks but not in the table: {missing if missing else 'none'}")
picks.to_csv(result_path("picks.csv"), index=False, date_format="%Y-%m-%dT%H:%M:%S.%f")
print("Pick format:", picks.head(), sep="\n")


BH?: 24 stations new to the table
HH?: 5 stations new to the table
29 unique stations
Station format:
     id   longitude   latitude  elevation_m
0  ARAT -121.853000  46.789000      1822.44
1  CARB -122.005410  46.988320       872.47
2  COPP -121.828750  46.797980      1895.00
3  CRBN -121.961026  46.988155       500.08
4  GNOB -121.914380  46.794140      1659.36

187787 picks from 2625 files: S 129712, P 58075
stations with picks but not in the table: none
Pick format:
     id                        timestamp type  prob
0   RCM 2023-08-23 19:20:59.320000+00:00    P   1.0
1   RCM 2023-08-23 19:23:21.930000+00:00    S   1.0
2   RCM 2023-08-23 19:23:39.880000+00:00    S   1.0
3   RCM 2023-08-23 19:24:11.970000+00:00    S   1.0
4  TAVI 2023-08-23 19:24:46.020000+00:00    S   1.0


In [7]:
## Region: the same lat 45-48, lon -123 to -120, 0-40 km box as the PyOcto notebook
## (OctoAssociator.from_area) and the DAS GaMMA run.
x0 = stations["longitude"].median()
y0 = stations["latitude"].median()
config = {}
config["center"] = (x0, y0)
config["xlim_degree"] = (-123, -120)
config["ylim_degree"] = (45, 48)

## projection to km
proj = Proj(f"+proj=aeqd +lon_0={config['center'][0]} +lat_0={config['center'][1]} +units=km")
stations[["x(km)", "y(km)"]] = stations.apply(lambda x: pd.Series(proj(longitude=x.longitude, latitude=x.latitude)), axis=1)
stations["z(km)"] = stations["elevation_m"].apply(lambda x: -x / 1e3)

### setting GMMA configs
config["use_dbscan"] = True
config["use_amplitude"] = False           # ELEP picks carry no amplitude
config["method"] = "BGMM"
if config["method"] == "BGMM":  ## BayesianGaussianMixture
    config["oversample_factor"] = 5
if config["method"] == "GMM":  ## GaussianMixture
    config["oversample_factor"] = 1
#!!Prior for (time and amplitude): set a larger value to prevent splitting events. A too large value will cause merging events.
# config["covariance_prior"] = [5.0, 2.0]

# earthquake location
config["vel"] = {"p": 6.0, "s": 6.0 / 1.75}       # only used for DBSCAN scaling once eikonal is set
config["dims"] = ["x(km)", "y(km)", "z(km)"]
config["x(km)"] = proj(longitude=config["xlim_degree"], latitude=[config["center"][1]] * 2)[0]
config["y(km)"] = proj(longitude=[config["center"][0]] * 2, latitude=config["ylim_degree"])[1]
config["z(km)"] = (0, 40)
config["bfgs_bounds"] = (
    (config["x(km)"][0] - 1, config["x(km)"][1] + 1),  # x
    (config["y(km)"][0] - 1, config["y(km)"][1] + 1),  # y
    (0, config["z(km)"][1] + 1),  # z
    (None, None),  # t
)

# DBSCAN:
##!!Truncate the picks into segments: change the dbscan_eps to balance speed and event splitting.
## Unlike the DAS run (channels 96 m apart, eps forced to 10 s), the broadband stations are 10-30 km
## apart, so estimate_eps() gives a sensible value and is used as in the GaMMA example.
config["dbscan_eps"] = estimate_eps(stations, config["vel"]["p"])
config["dbscan_min_samples"] = 3

## using Eikonal for 1D velocity model: IASP91 down to the bottom of the search depth
velocity_model = pd.read_csv(iasp91_csv)                      # columns depth, r, vp, vs
velocity_model = velocity_model[velocity_model["depth"] <= config["z(km)"][1]].copy()
# IASP91 repeats the depth at layer boundaries; np.interp needs increasing depths, so nudge duplicates
velocity_model["depth"] += velocity_model.groupby("depth").cumcount() * 1e-3
vel = {"z": velocity_model["depth"].values, "p": velocity_model["vp"].values, "s": velocity_model["vs"].values}
h = 1.0
config["eikonal"] = {"vel": vel, "h": h, "xlim": config["x(km)"], "ylim": config["y(km)"], "zlim": config["z(km)"]}

# set number of cpus
config["ncpu"] = min(16, os.cpu_count())

##!!Post filtering (indepent of gmm): change these parameters to filter out associted picks with large errors
## Same acceptance as the PyOcto notebook's FULL-network run: n_picks=10, n_p_picks=3, n_s_picks=3.
## The 5-station run below lowers min_picks_per_eq to 6 in its own config_5.
config["min_picks_per_eq"] = 10
config["min_p_picks_per_eq"] = 3
config["min_s_picks_per_eq"] = 3
config["max_sigma11"] = 2.0  # second
config["max_sigma22"] = 1.0  # log10(m/s)
config["max_sigma12"] = 1.0  # covariance

for k, v in config.items():
    if k == "eikonal":
        print(f"{k}: IASP91 {len(vel['z'])} layers, h={h} km, zlim={v['zlim']}")
    else:
        print(f"{k}: {v}")


center: (-121.88423, 46.83557)
xlim_degree: (-123, -120)
ylim_degree: (45, 48)
use_dbscan: True
use_amplitude: False
method: BGMM
oversample_factor: 5
vel: {'p': 6.0, 's': 3.4285714285714284}
dims: ['x(km)', 'y(km)', 'z(km)']
x(km): (-85.1181327472162, 143.7322669593862)
y(km): (-204.0230732342197, 129.46014427772832)
z(km): (0, 40)
bfgs_bounds: ((-86.1181327472162, 144.7322669593862), (-205.0230732342197, 130.46014427772832), (0, 41), (None, None))
dbscan_eps: 4.390794089377598
dbscan_min_samples: 3
eikonal: IASP91 39 layers, h=1.0 km, zlim=(0, 40)
ncpu: 16
min_picks_per_eq: 10
min_p_picks_per_eq: 3
min_s_picks_per_eq: 3
max_sigma11: 2.0
max_sigma22: 1.0
max_sigma12: 1.0


## 2. Association with GaMMA

In [8]:
t0 = time.time()
event_idx0 = 0  ## current earthquake index
assignments = []
events, assignments = association(picks, stations, config, event_idx0, config["method"])
event_idx0 += len(events)
print(f"{len(events)} events from {len(assignments)} associated picks in {time.time() - t0:.0f} s")

## create catalog
events = pd.DataFrame(events)
if len(events):
    events[["longitude", "latitude"]] = events.apply(lambda x: pd.Series(proj(longitude=x["x(km)"], latitude=x["y(km)"], inverse=True)), axis=1)
    events["depth_km"] = events["z(km)"]
    events.to_csv(result_path("gamma_events.csv"), index=False,
                  float_format="%.3f",
                  date_format="%Y-%m-%dT%H:%M:%S.%f")

## add assignment to picks
assignments = pd.DataFrame(assignments, columns=["pick_index", "event_index", "gamma_score"])
picks = picks.join(assignments.set_index("pick_index")).fillna(-1).astype({"event_index": int})
picks.rename(columns={"id": "station_id", "timestamp": "phase_time", "type": "phase_type", "prob": "phase_score"}, inplace=True)
picks.to_csv(result_path("gamma_picks.csv"), index=False,
             date_format="%Y-%m-%dT%H:%M:%S.%f")

if len(events):
    n_by_phase = picks[picks["event_index"] >= 0].groupby(["event_index", "phase_type"]).size().unstack(fill_value=0)
    events = events.merge(n_by_phase.rename(columns={"P": "n_p", "S": "n_s"}), left_on="event_index", right_index=True, how="left")
    cols = ["event_index", "time", "longitude", "latitude", "depth_km", "num_picks", "n_p", "n_s", "sigma_time", "gamma_score"]
    events["time"] = pd.to_datetime(events["time"], utc=True)
    per_day = events.set_index("time").resample("1D").size()
    print(f"events per day: median {per_day.median():.0f}, max {per_day.max()} on {per_day.idxmax().date()}; "
          f"{(events['n_p'] == 0).sum()} S-only events, {(events['n_s'] == 0).sum()} P-only events")
    with pd.option_context("display.width", 200, "display.max_columns", 20):
        print(events[[c for c in cols if c in events.columns]].head(30).to_string(index=False))


Eikonal Solver: 
Iter 0, error = 999.828
Iter 1, error = 0.000
Time: 5.274
Eikonal Solver: 
Iter 0, error = 999.702
Iter 1, error = 0.000
Time: 0.049


Clustering (eps=3.66): 100%|██████████| 10963/10963 [00:03<00:00, 3199.02it/s]


Associating 10962 clusters with 16 CPUs
...........................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................
Associated 100 events
..............................................

In [9]:
gamma_events = pd.read_csv(result_path("gamma_events.csv"))
gamma_events


,time,magnitude,sigma_time,sigma_amp,cov_time_amp,gamma_score,num_picks,num_p_picks,num_s_picks,event_index,x(km),y(km),z(km),longitude,latitude,depth_km
0,2023-10-31T19:18:27.266,999,0.173,0,0,28.0,27,13,14,1,10.118,0.226,1.021,-121.752,46.838,1.021
1,2023-08-24T11:46:34.912,999,0.276,0,0,20.0,20,10,10,6,9.899,3.499,0.987,-121.754,46.867,0.987
2,2023-09-17T11:29:48.674,999,0.575,0,0,41.0,38,27,11,26,-2.436,13.869,9.505,-121.916,46.960,9.505
3,2023-08-27T12:41:20.686,999,0.186,0,0,51.0,51,25,26,30,-4.116,-6.937,10.800,-121.938,46.773,10.800
4,2023-09-13T06:02:29.968,999,0.146,0,0,35.0,35,17,18,9,-4.023,-6.457,10.580,-121.937,46.777,10.580
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1379,2023-10-21T15:33:24.082,999,0.743,0,0,16.0,16,13,3,1367,7.970,3.774,0.167,-121.780,46.869,0.167
1380,2023-09-28T05:23:59.067,999,0.545,0,0,11.0,11,5,6,1373,10.172,3.348,0.000,-121.751,46.866,0.000
1381,2023-11-20T12:54:58.099,999,0.203,0,0,21.0,21,10,11,1378,13.183,-43.499,17.778,-121.713,46.444,17.778
1382,2023-10-31T00:36:14.901,999,0.784,0,0,13.0,13,5,8,1368,12.042,0.943,0.000,-121.726,46.844,0.000


## 3. Check against the PyOcto catalogue

In [10]:
## Match the two station catalogues on origin time. A GaMMA event is matched to the nearest PyOcto
## event within `dt_max` seconds; the separation is reported for the matches. Written as a function
## so the 5-station run below is checked the same way.
def compare_to_pyocto(gamma_events, pyocto_csv, dt_max=10.0, label=""):
    pyocto_events = pd.read_csv(pyocto_csv)
    pyocto_events["time"] = pd.to_datetime(pyocto_events["time"], utc=True)
    pyocto_events = pyocto_events.sort_values("time").reset_index(drop=True)

    g = gamma_events.copy()
    g["time"] = pd.to_datetime(g["time"], utc=True)
    g = g.sort_values("time").reset_index(drop=True)

    # merge_asof keeps only the GaMMA `time`, so carry PyOcto's origin time along under its own name.
    m = pd.merge_asof(g, pyocto_events.assign(time_octo=pyocto_events["time"]), on="time", direction="nearest",
                      tolerance=pd.Timedelta(seconds=dt_max), suffixes=("_gamma", "_octo"))
    matched = m[m["idx"].notna()].copy()
    matched["dt_s"] = (matched["time"] - matched["time_octo"]).dt.total_seconds()
    dx, dy = proj(longitude=matched["longitude_octo"].values, latitude=matched["latitude_octo"].values)
    matched["dist_km"] = np.hypot(matched["x(km)"] - dx, matched["y(km)"] - dy)
    matched["ddepth_km"] = matched["depth_km"] - matched["depth"]

    print(f"=== {label} ===" if label else "")
    print(f"GaMMA : {len(g)} events")
    print(f"PyOcto: {len(pyocto_events)} events   ({os.path.basename(pyocto_csv)})")
    print(f"matched within {dt_max:.0f} s: {len(matched)} "
          f"({len(matched) / max(len(g), 1):.0%} of GaMMA, {matched['idx'].nunique() / max(len(pyocto_events), 1):.0%} of PyOcto)")
    print(f"GaMMA-only : {len(g) - len(matched)}")
    print(f"PyOcto-only: {len(pyocto_events) - matched['idx'].nunique()}")
    if len(matched):
        print("\nmatched events, GaMMA minus PyOcto (median / 90th pct of |difference|):")
        for c, unit in [("dt_s", "s"), ("dist_km", "km"), ("ddepth_km", "km")]:
            print(f"  {c:10s} {matched[c].abs().median():6.2f} / {matched[c].abs().quantile(0.9):6.2f} {unit}")
    print(f"\npicks per event: GaMMA median {g['num_picks'].median():.0f}, PyOcto median {pyocto_events['picks'].median():.0f}\n")
    return matched

matched_all = compare_to_pyocto(gamma_events, os.path.join(pyocto_dir, "events_detect_octo_staonly.csv"),
                                dt_max=10.0, label="all 29 stations")


=== all 29 stations ===
GaMMA : 1384 events
PyOcto: 1495 events   (events_detect_octo_staonly.csv)
matched within 10 s: 1189 (86% of GaMMA, 78% of PyOcto)
GaMMA-only : 195
PyOcto-only: 322

matched events, GaMMA minus PyOcto (median / 90th pct of |difference|):
  dt_s         0.28 /   0.73 s
  dist_km      0.94 /   3.95 km
  ddepth_km    0.62 /   4.35 km

picks per event: GaMMA median 15, PyOcto median 17



## 4. Strict parameters - 5 stations

Same as the "strict parameters - 5 station" section of the PyOcto notebook: only the five stations
nearest the DAS cable (CC KAUT, UW LON, CC MILD, CC PARA, UW RER), same acceptance as PyOcto's
5-station run (`min_picks_per_eq=6, min_p_picks_per_eq=3, min_s_picks_per_eq=3`; the full network
above uses 10), same box and velocity model.
The DBSCAN eps is re-estimated from the 5-station spacing, as `estimate_eps` would do for any table.
Outputs go to `gamma_events_5sta.csv` / `gamma_picks_5sta.csv` in the same results folder and are
checked against PyOcto's `events_detect_octo_staonly_only_KAUT-LON-MILK-PARA-RER-11032026_strict.csv`.


In [11]:
# CC-KAUT, UW-LON, CC-MILD, CC-PARA, UW-RER
small_station_ids = ["KAUT", "LON", "MILD", "PARA", "RER"]
small_stations = stations[stations["id"].isin(small_station_ids)].reset_index(drop=True)

# Picks reloaded from picks.csv (GaMMA format, written before the full association), so this cell
# does not depend on the renamed/joined `picks` left behind by the full run.
small_picks = pd.read_csv(result_path("picks.csv"))
small_picks["timestamp"] = pd.to_datetime(small_picks["timestamp"], utc=True, format="ISO8601")
small_picks = small_picks[small_picks["id"].isin(small_station_ids)].sort_values("timestamp").reset_index(drop=True)

config_5 = dict(config)
config_5["min_picks_per_eq"] = 6        # PyOcto 5-station run uses n_picks=6 (full network: 10)
config_5["dbscan_eps"] = estimate_eps(small_stations, config["vel"]["p"])
print(f"{len(small_stations)} stations, {len(small_picks)} picks: "
      + ", ".join(f"{k} {v}" for k, v in small_picks["type"].value_counts().items()))
print(f"dbscan_eps: {config_5['dbscan_eps']:.2f} s (full table used {config['dbscan_eps']:.2f} s); "
      f"min_picks_per_eq {config_5['min_picks_per_eq']} (full table used {config['min_picks_per_eq']})")
small_stations


5 stations, 39483 picks: S 28476, P 11007
dbscan_eps: 2.62 s (full table used 4.39 s); min_picks_per_eq 6 (full table used 10)


,id,longitude,latitude,elevation_m,x(km),y(km),z(km)
0,KAUT,-121.857381,46.730263,689.00,2.052283,-11.706271,-0.68900
1,MILD,-121.775680,46.808100,2015.59,8.285389,-3.048044,-2.01559
2,PARA,-121.742195,46.786441,1651.18,10.845577,-5.451729,-1.65118
3,LON,-121.809601,46.750599,853.00,5.702342,-9.443247,-0.85300
4,RER,-121.842500,46.818580,1751.10,3.184544,-1.887889,-1.75110


In [12]:
t0 = time.time()
events_5, assignments_5 = association(small_picks, small_stations, config_5, 0, config_5["method"])
print(f"{len(events_5)} events from {len(assignments_5)} associated picks in {time.time() - t0:.0f} s")

events_5 = pd.DataFrame(events_5)
if len(events_5):
    events_5[["longitude", "latitude"]] = events_5.apply(lambda x: pd.Series(proj(longitude=x["x(km)"], latitude=x["y(km)"], inverse=True)), axis=1)
    events_5["depth_km"] = events_5["z(km)"]
    events_5.to_csv(result_path("gamma_events_5sta.csv"), index=False,
                    float_format="%.3f", date_format="%Y-%m-%dT%H:%M:%S.%f")

assignments_5 = pd.DataFrame(assignments_5, columns=["pick_index", "event_index", "gamma_score"])
small_picks = small_picks.join(assignments_5.set_index("pick_index")).fillna(-1).astype({"event_index": int})
small_picks.rename(columns={"id": "station_id", "timestamp": "phase_time", "type": "phase_type", "prob": "phase_score"}, inplace=True)
small_picks.to_csv(result_path("gamma_picks_5sta.csv"), index=False, date_format="%Y-%m-%dT%H:%M:%S.%f")

if len(events_5):
    n_by_phase = small_picks[small_picks["event_index"] >= 0].groupby(["event_index", "phase_type"]).size().unstack(fill_value=0)
    events_5 = events_5.merge(n_by_phase.rename(columns={"P": "n_p", "S": "n_s"}), left_on="event_index", right_index=True, how="left")
    cols = ["event_index", "time", "longitude", "latitude", "depth_km", "num_picks", "n_p", "n_s", "sigma_time", "gamma_score"]
    with pd.option_context("display.width", 200, "display.max_columns", 20):
        print(events_5[[c for c in cols if c in events_5.columns]].head(30).to_string(index=False))


Eikonal Solver: 
Iter 0, error = 999.828
Iter 1, error = 0.000
Time: 0.024
Eikonal Solver: 
Iter 0, error = 999.702
Iter 1, error = 0.000
Time: 0.023


Clustering (eps=2.18): 100%|██████████| 3070/3070 [00:00<00:00, 21982.80it/s]


Associating 3069 clusters with 16 CPUs
.................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

In [13]:
gamma_events_5 = pd.read_csv(result_path("gamma_events_5sta.csv"))
matched_5 = compare_to_pyocto(gamma_events_5,
                              os.path.join(pyocto_dir, "events_detect_octo_staonly_only_KAUT-LON-MILK-PARA-RER-11032026_strict.csv"),
                              dt_max=10.0, label="5 stations, strict")
gamma_events_5


=== 5 stations, strict ===
GaMMA : 266 events
PyOcto: 354 events   (events_detect_octo_staonly_only_KAUT-LON-MILK-PARA-RER-11032026_strict.csv)
matched within 10 s: 229 (86% of GaMMA, 65% of PyOcto)
GaMMA-only : 37
PyOcto-only: 125

matched events, GaMMA minus PyOcto (median / 90th pct of |difference|):
  dt_s         0.08 /   0.30 s
  dist_km      1.40 /   3.58 km
  ddepth_km    0.66 /   2.68 km

picks per event: GaMMA median 7, PyOcto median 8



,time,magnitude,sigma_time,sigma_amp,cov_time_amp,gamma_score,num_picks,num_p_picks,num_s_picks,event_index,x(km),y(km),z(km),longitude,latitude,depth_km
0,2023-09-30T13:55:56.824,999,0.188,0,0,6.0,6,3,3,0,-2.514,-13.051,3.053,-121.917,46.718,3.053
1,2023-10-01T03:12:25.029,999,0.114,0,0,9.0,9,4,5,3,11.765,1.812,0.556,-121.730,46.852,0.556
2,2023-10-21T07:54:37.694,999,0.316,0,0,7.0,7,3,4,1,7.073,-4.834,4.363,-121.792,46.792,4.363
3,2023-11-20T01:26:10.041,999,0.051,0,0,6.0,6,3,3,20,-3.246,5.780,14.383,-121.927,46.888,14.383
4,2023-11-20T20:42:43.973,999,0.153,0,0,9.0,9,4,5,9,-6.332,3.585,9.028,-121.967,46.868,9.028
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,2023-10-28T10:34:02.501,999,0.605,0,0,6.0,6,3,3,262,-6.563,0.917,14.631,-121.970,46.844,14.631
262,2023-09-20T07:18:11.998,999,0.353,0,0,8.0,8,4,4,256,7.094,-1.438,3.879,-121.791,46.823,3.879
263,2023-10-23T07:13:50.104,999,0.245,0,0,6.0,6,3,3,258,16.113,3.174,15.272,-121.673,46.864,15.272
264,2023-10-07T01:25:13.766,999,0.062,0,0,7.0,7,4,3,265,9.908,2.807,0.000,-121.754,46.861,0.000
